In [50]:
import google.auth
import numpy as np
import pandas as pd
import pygris 
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase

In [2]:

import os
from typing import List, Optional, Union
import pyarrow.dataset as ds
from google.cloud import storage

In [3]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas

In [4]:
# uv add pandas pyarrow gcsfs google-cloud-storage fsspec
# uv pip install pytidycensus

In [5]:
@cache
def gcs_pandas():
    return GCSPandas()

In [6]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

## Census Block

In [7]:
analysis_year = 2023

In [51]:

blocks = pygris.blocks(
    state="06",     # California
    county="067",   # Sacramento, e.g.
    year=2020,
    cache=True
)


In [52]:
blocks.head(1)

,STATEFP20,COUNTYFP20,TRACTCE20,BLOCKCE20,GEOID20,NAME20,MTFCC20,UR20,UACE20,UATYPE20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,HOUSING20,POP20,geometry
615,06,067,009635,2008,060670096352008,Block 2008,G5040,R,None,None,S,28414,0,+38.3798570,-121.4533728,12,22,"POLYGON ((-121.45453 38.37933, -121.45437 38.38054, -121.45228 38.38053, -121.45227 38.37921, -121.45454 38.37918, -121.45453 38.37933))"


In [53]:
counties = tc.get_decennial(
        geography="county",
        variables="P1_001N",
        state="06",
        year=2020,
        geometry=False
    )

Getting data from the 2020 decennial Census
Using the PL 94-171 Redistricting Data Summary File


/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/pytidycensus/decennial.py:429: UserWarning: Note: 2020 decennial Census data use differential privacy, a technique that introduces errors into data to preserve respondent confidentiality. Small counts should be interpreted with caution. See https://www.census.gov/library/fact-sheets/2021/protecting-the-confidentiality-of-the-2020-census-redistricting-data.html for additional guidance.
  warnings.warn(


In [12]:
counties.head(2)

,GEOID,P1_001N,state,county,NAME
0,06001,1682353,06,001,"Alameda County, California"
1,06003,1204,06,003,"Alpine County, California"


In [58]:

def load_ca_blocks(year=2020, variables=["P1_001N"], geometry=True):
    """
    Load all census blocks for California (2020 PL94 decennial).
    Downloads one county at a time because the API does not allow
    state-level block requests.
    """
    
    # Get list of CA counties
    counties = tc.get_decennial(
        geography="county",
        variables=variables,
        state="06",
        year=year,
        geometry=False
    )["county"].unique()
    
    all_blocks = []

    for c in ['001']:
        df = pygris.blocks(
        state="06",     # California
        county="067",   # Sacramento, e.g.
        year=2020,
        cache=True
       )

        all_blocks.append(df)

    all_blocks = to_snakecase(pd.concat(all_blocks, ignore_index=True))
    
    """
    all_blocks = all_blocks[["name_y",
                            "tract",
                            "block_group",
                            "geometry",
                            "estimate",
                            "variable"]]
    """
    # Reproject
    all_blocks = all_blocks.to_crs(geography_utils.CA_NAD83Albers_ft)

    # Buffer
    all_blocks["b250"] = all_blocks.buffer(250)
    all_blocks = all_blocks.drop(columns = ["geometry"])
    return all_blocks


In [59]:
alameda_blocks_gdf = load_ca_blocks()

Getting data from the 2020 decennial Census
Using the PL 94-171 Redistricting Data Summary File


/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/pytidycensus/decennial.py:429: UserWarning: Note: 2020 decennial Census data use differential privacy, a technique that introduces errors into data to preserve respondent confidentiality. Small counts should be interpreted with caution. See https://www.census.gov/library/fact-sheets/2021/protecting-the-confidentiality-of-the-2020-census-redistricting-data.html for additional guidance.
  warnings.warn(


In [61]:
alameda_blocks_gdf.head(1).drop(columns = ["b250"])

,statefp20,countyfp20,tractce20,blockce20,geoid20,name20,mtfcc20,ur20,uace20,uatype20,funcstat20,aland20,awater20,intptlat20,intptlon20,housing20,pop20
0,06,067,009635,2008,060670096352008,Block 2008,G5040,R,None,None,S,28414,0,+38.3798570,-121.4533728,12,22


In [60]:
len(alameda_blocks_gdf)

18522

## Public Road Functional Classification
**Amanda** Need to fix: URL maxes out at 2000 rows when there are thousands more. 

In [17]:
# https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/query?where=1%3D1&outFields=F_System,Shape__Length,Caltrans_District,RouteID&outSR=102600&f=json
# "https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"

In [18]:
public_road_url = "https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/query?where=1%3D1&outFields=F_System,Shape__Length,Caltrans_District,RouteID&outSR=102600&f=json"

In [19]:
public_road_gdf = to_snakecase(gpd.read_file(public_road_url))

In [21]:
public_road_gdf.geometry.type.unique()

array(['MultiLineString', 'LineString'], dtype=object)

In [25]:
public_road_gdf.sample(10).drop(columns = ["geometry"])

,f_system,shape__length,caltrans_district,routeid
1745,7,231.82,6,ALY_811605_P
618,7,348.37,7,ALY_809394_P
1337,7,101.27,6,ALY_810990_P
1531,7,154.65,6,ALY_811310_P
1768,7,53.04,6,ALY_811729_P
719,7,22.85,5,ALY_809973_P
938,7,143.13,6,ALY_810086_P
108,7,114.72,5,ALY_800027_P
1846,7,140.01,7,ALY_811960_P
102,7,93.83,5,ALY_800021_P


In [26]:
public_road_gdf.f_system.unique()

array([4, 3, 5, 2, 7])

In [27]:
public_road_gdf2 = public_road_gdf.loc[~public_road_gdf["f_system"].isin([1,2])]

In [28]:
len(public_road_gdf2), len(public_road_gdf)

(1986, 2000)

In [30]:
public_road_gdf2 = public_road_gdf2.set_crs(geography_utils.CA_NAD83Albers_ft)

In [62]:
public_road_gdf2["b50"] = public_road_gdf2.geometry.buffer(50)

In [63]:
public_road_gdf2 = public_road_gdf2.drop(columns = ["geometry"])

In [65]:
public_road_gdf2 = public_road_gdf2.set_geometry("b50")

## TIMS Data

In [31]:
def _parse_gcs_path(gcs_path: str):
    """
    Split a GCS URL into (bucket, prefix) without leading 'gs://'.
    """
    if not gcs_path.startswith("gs://"):
        raise ValueError(f"Expected a 'gs://' path, got: {gcs_path}")
    no_scheme = gcs_path[5:]
    bucket, *rest = no_scheme.split("/", 1)
    prefix = rest[0] if rest else ""
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    return bucket, prefix

In [32]:
def list_gcs_files(gcs_folder: str, extensions: Optional[List[str]] = None) -> list:
    """
    List all files in a GCS 'folder' (prefix). Optionally filter by extensions.
    Returns full 'gs://...' URIs.
    """
    bucket_name, prefix = _parse_gcs_path(gcs_folder)
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    uris: List[str] = []
    for blob in client.list_blobs(bucket_name, prefix=prefix):
        # Skip "directory placeholders"
        name = blob.name
        if name.endswith("/"):
            continue
        if extensions:
            if not any(name.lower().endswith(ext.lower()) for ext in extensions):
                continue
        uris.append(f"gs://{bucket_name}/{name}")

    return sorted(uris)

In [33]:

try:
    import geopandas as gpd
    _HAS_GPD = True
except Exception:
    _HAS_GPD = False


In [34]:
def concat_gcs_folder(
    gcs_folder: str = "gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset: bool = True,
    file_types: Optional[List[str]] = None,
    geometry: bool = False,
    dtype_overrides: Optional[dict] = None,
    use_threads: bool = True,
) -> Union[pd.DataFrame, "gpd.GeoDataFrame"]:
    """
    Concatenate all files in a GCS folder into a single DataFrame.

    Parameters
    ----------
    gcs_folder : str
        GCS folder URL to read.
    prefer_arrow_dataset : bool
        If True and all files are parquet, use pyarrow.dataset for fast ingestion.
    file_types : List[str] or None
        Limit to these extensions (e.g., ["parquet","csv","feather","geojson"]).
        If None, auto-detect common formats.
    geometry : bool
        If True and reading GeoJSON, return a GeoDataFrame (requires geopandas).
    dtype_overrides : dict or None
        Dict of column dtypes to enforce after read (e.g., {"GEOID":"string"}).
    use_threads : bool
        Pass-through to pandas readers to enable multi-threaded parsing when supported.

    Returns
    -------
    DataFrame or GeoDataFrame
    """
    # Default formats we’ll support
    if file_types is None:
        file_types = ["parquet", "csv", "feather", "geojson", "json"]  # last two for geospatial or line-delimited JSON

    files = list_gcs_files(gcs_folder, extensions=[f".{ext}" for ext in file_types])

    if not files:
        raise FileNotFoundError(f"No files found under: {gcs_folder} with types {file_types}")

    # If all files are parquet and arrow dataset is preferred → use PA dataset
    all_parquet = all(f.lower().endswith(".parquet") for f in files)
    if prefer_arrow_dataset and all_parquet:
        # Arrow can read the entire folder as one dataset (partition-aware)
        dataset = ds.dataset(gcs_folder, format="parquet")
        table = dataset.to_table(use_threads=use_threads)
        df = table.to_pandas(types_mapper=pd.ArrowDtype)
        if dtype_overrides:
            df = df.astype(dtype_overrides, errors="ignore")
        return df

    # Otherwise, iterate by type and read via pandas / geopandas
    frames: List[Union[pd.DataFrame, "gpd.GeoDataFrame"]] = []

    for uri in files:
        lower = uri.lower()
        if lower.endswith(".parquet"):
            # pandas supports GCS via fsspec/gcsfs
            frames.append(pd.read_parquet(uri))
        elif lower.endswith(".feather"):
            frames.append(pd.read_feather(uri))
        elif lower.endswith(".csv"):
            frames.append(pd.read_csv(uri, low_memory=False))
        elif lower.endswith(".geojson") or (lower.endswith(".json") and "geo" in os.path.basename(uri).lower()):
            if not _HAS_GPD:
                raise ImportError("geopandas not installed—install it or set geometry=False.")
            gdf = gpd.read_file(uri)
            frames.append(gdf)
        else:
            # Skip unknown formats
            print(f"[concat_gcs_folder] Skipping unsupported file: {uri}")

    if not frames:
        raise FileNotFoundError(f"Found files, but none were readable with the allowed types: {file_types}")

    # Concatenate; if any GeoDataFrames present, upcast to GeoDataFrame
    if _HAS_GPD and any(isinstance(f, gpd.GeoDataFrame) for f in frames):
        df = pd.concat(frames, ignore_index=True)
        # Rebuild geometry column if lost during concat (rare)
        if "geometry" in df.columns and not isinstance(df, gpd.GeoDataFrame):
            df = gpd.GeoDataFrame(df, geometry="geometry", crs=frames[0].crs if hasattr(frames[0], "crs") else None)
    else:
        df = pd.concat(frames, ignore_index=True)

    if dtype_overrides:
        df = df.astype(dtype_overrides, errors="ignore")

    df = to_snakecase(df)
    return df

In [35]:

tims_df = concat_gcs_folder(
    gcs_folder="gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset=True
)


In [36]:
tims_df.head(2)

,case_id,accident_year,proc_date,juris,collision_date,collision_time,officer_id,reporting_district,day_of_week,chp_shift,population,cnty_city_loc,special_cond,beat_type,chp_beat_type,city_division_lapd,chp_beat_class,beat_number,primary_rd,secondary_rd,distance,direction,intersection,weather_1,weather_2,state_hwy_ind,caltrans_county,caltrans_district,state_route,route_suffix,postmile_prefix,postmile,location_type,ramp_intersection,side_of_hwy,tow_away,collision_severity,number_killed,number_injured,party_count,primary_coll_factor,pcf_code_of_viol,pcf_viol_category,pcf_violation,pcf_viol_subsection,hit_and_run,type_of_collision,mviw,ped_action,road_surface,road_cond_1,road_cond_2,lighting,control_device,chp_road_type,pedestrian_accident,bicycle_accident,motorcycle_accident,truck_accident,not_private_property,alcohol_involved,stwd_vehtype_at_fault,chp_vehtype_at_fault,count_severe_inj,count_visible_inj,count_complaint_pain,count_ped_killed,count_ped_injured,count_bicyclist_killed,count_bicyclist_injured,count_mc_killed,count_mc_injured,primary_ramp,secondary_ramp,latitude,longitude,county,city,point_x,point_y
0,81456383,2021,4/22/2021,107,3/14/2021,1235,43,Liver,7,5,5,107,0,0,0,NaN,0,4,HOLMES ST,LEXINGTON WY,599.00,S,N,B,-,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,4,0,1,2,A,-,8,22107.00,NaN,N,C,G,A,B,H,-,A,D,0,NaN,Y,NaN,NaN,Y,NaN,L,4,0,0,1,0,0,0,1,0,0,-,-,37.65,-121.78,ALAMEDA,LIVERMORE,-121.78,37.65
1,81456384,2021,4/22/2021,107,3/16/2021,1255,79,Liver,2,5,5,107,0,0,0,NaN,0,1,L ST,LINDEN ST,105.00,N,N,A,-,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3,0,1,2,A,-,9,21804.00,A,N,D,C,A,A,H,-,A,D,0,NaN,NaN,NaN,NaN,Y,NaN,A,1,0,1,0,0,0,0,0,0,0,-,-,37.69,-121.77,ALAMEDA,LIVERMORE,-121.77,37.69


In [37]:
tims_gdf = gpd.GeoDataFrame(
    tims_df, geometry=gpd.points_from_xy(tims_df.point_x, tims_df.point_y), crs=geography_utils.CA_NAD83Albers_ft
)

In [38]:
tims_gdf.shape

(486776, 81)

In [68]:
type(tims_gdf)

geopandas.geodataframe.GeoDataFrame

In [74]:
tims_gdf.head(100).explore()

In [69]:
type(public_road_gdf2)

geopandas.geodataframe.GeoDataFrame

In [70]:
public_road_gdf2.shape

(1986, 5)

In [71]:
public_road_gdf2.head().explore()

## Overlay TIMS with Public Road Functional Classification data

In [66]:
tims_public_road = (
        tims_gdf.sjoin(public_road_gdf2, how="inner", predicate="intersects")
        .reset_index(drop=True)
        .drop(columns=["index_right"])
    )

In [67]:
len(tims_public_road)

0